# Grouped participant validation

Use five development folds with a fresh conventional Adam GRU in each fold.
All 26 development participants are validated once. Final test participants remain reserved.
Read docs/grouped-validation.md before enabling preparation or training.
Each validation fold also selects its checkpoint; these are development results.
Report export runs planning and saved-result inspection, never preparation or training.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("gru_suite_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/gru-grouped-validation.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/gru-grouped-validation"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Run preparation: %s", RUN_PREPARATION)
logger.info("Run training: %s", RUN_TRAINING)

## Optional environment setup

In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Plan the participant folds

This command reads metadata only. It does not read feature observations or train a model.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
    --config configs/gru-grouped-validation.yaml --plan
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Fold planning failed")

## Prepare derived fold tables without training

Enable RUN_PREPARATION and execute the next cell once.
This copies prepared development features and freezes configurations and hashes.
It fits no scaler or model. Existing output directories are refused.


In [ ]:
if RUN_PREPARATION:
    !uv run --locked --group gru --group notebook python \
        -m sleep_classifier.grouped_validation \
        --config configs/gru-grouped-validation.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Grouped preparation failed")
else:
    logger.info("Preparation skipped")

## Train the five models when ready

Enable RUN_TRAINING after preparing the folds. Execute this cell interactively.
The maximum is five fits and 100 epochs. Early stopping can reduce that total.
The runner saves progress after each fold. It refuses previous training results.


In [ ]:
if RUN_TRAINING:
    !uv run --locked --group gru --group notebook python -u \
        -m sleep_classifier.grouped_validation \
        --config configs/gru-grouped-validation.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Grouped training failed; inspect the partial summary")
else:
    logger.info("Training skipped: enable RUN_TRAINING and execute this cell when ready")

## Inspect saved development results

Read pooled epoch scores alongside fold and participant scores.
Checkpoint selection used the same validation folds. These are not independent outer-test results.


In [ ]:
import pandas as pd
import yaml

config_path = ROOT / "configs/gru-grouped-validation.yaml"
settings = yaml.safe_load(config_path.read_text())
summary_path = (config_path.parent / settings["output_dir"] / "summary.json").resolve()
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    logger.info("Grouped experiment status: %s", summary["status"])
    results = pd.DataFrame(summary["results"])
    if not results.empty:
        display(results[["fold", "macro_f1", "accuracy", "best_epoch"]])
        for result in summary["results"]:
            logger.info("Participant results for %s", result["fold"])
            display(pd.DataFrame(result["participants"]).T)
    if "aggregate" in summary:
        aggregate = summary["aggregate"]
        logger.info("Aggregate development results: %s", aggregate)
else:
    logger.info("No grouped training results exist. Plan/report execution did not train.")

In [ ]:
if summary_path.exists() and summary["results"]:
    import matplotlib.pyplot as plt
    import numpy as np

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    results.set_index("fold")[["macro_f1", "accuracy"]].plot.bar(ax=axes[0], ylim=(0, 1))
    axes[0].set(xlabel="Development fold", ylabel="Selected-checkpoint score")
    if "aggregate" in summary:
        counts = np.asarray(summary["aggregate"]["confusion_matrix"])
        fractions = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
        axes[1].imshow(fractions, vmin=0, vmax=1, cmap="Blues")
        stages = ["Wake", "Light", "Deep", "REM"]
        axes[1].set_xticks(range(4), stages)
        axes[1].set_yticks(range(4), stages)
        axes[1].set(xlabel="Predicted stage", ylabel="True stage")
        for row in range(4):
            for column in range(4):
                axes[1].text(column, row, f"{fractions[row, column]:.0%}",
                             ha="center", va="center",
                             color="white" if fractions[row, column] > 0.5 else "black")
    figure.tight_layout()
    plt.show()

## Export HTML

In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/11_grouped_validation.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Report export failed")